In [2]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Load data
Data needed: 011626_demographics.csv, 011626_control_condition_df.csv, 011626_adhd_condition_df.csv

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_demographics.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
demographics  <- read_csv(name_of_file_in_bucket)

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_control_condition_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
control_condition_df  <- read_csv(name_of_file_in_bucket)

In [ ]:
# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_adhd_condition_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
adhd_condition_df  <- read_csv(name_of_file_in_bucket)

## Process data for descriptive analysis

In [ ]:
# Merge ADHD and control condition_dfs
condition_df <- bind_rows(adhd_condition_df, control_condition_df)

# Filter df to retain rows with person_ids in demographics
condition_df <- condition_df %>%
    filter(person_id %in% demographics$person_id)

In [ ]:
head(condition_df)

In [ ]:
# Define functions for extracting dxs

# Define function: extract_ADHD_subtype()
extract_ADHD_subtype <- function(data, id_column, dx_column) {

  # Define list of ADHD diagnoses
    dx_list_ADHD <- c("Attention deficit hyperactivity disorder, combined type",
                      "Attention deficit hyperactivity disorder, predominantly inattentive type",
                      "Attention deficit hyperactivity disorder, predominantly hyperactive impulsive type",
                      "Attention deficit hyperactivity disorder",
                      "Adult attention deficit hyperactivity disorder",
                      "Child attention deficit disorder",
                      "Undifferentiated attention deficit disorder")
  
  # Define buckets for ADHD subtypes  
      ADHD_H <- "Attention deficit hyperactivity disorder, predominantly hyperactive impulsive type"
      ADHD_U <- c("Attention deficit hyperactivity disorder",
                  "Adult attention deficit hyperactivity disorder",
                  "Child attention deficit disorder",
                  "Undifferentiated attention deficit disorder"
      )
      ADHD_PI <- "Attention deficit hyperactivity disorder, predominantly inattentive type"
      ADHD_C <- "Attention deficit hyperactivity disorder, combined type"
  
  data %>%
    distinct() %>%
    select({{id_column}},{{dx_column}}) %>%
    group_by({{id_column}}) %>%

    mutate( ADHD = 
      case_when(
        # ADHD = "None" if no ADHD diagnosis is present
        !any({{dx_column}} %in% dx_list_ADHD) ~ "None", 
        # ADHD = "Combined" if combined diagnosis is present OR
        any({{dx_column}} %in% ADHD_C) |
          # If hyperactive and inattentive diagnoses co-occur
          (any({{dx_column}} %in% ADHD_PI) & any({{dx_column}} %in% ADHD_H)) ~ "Combined",
        # ADHD = "Inattentive" if inattentive is present
        any({{dx_column}} %in% ADHD_PI) ~ "Inattentive",
        # ADHD = "Hyperactive" if hyperactive is present
        any({{dx_column}} %in% ADHD_H) ~ "Hyperactive",
        # ADHD = "Unspecified" if only unspecified diagnoses are present
        any({{dx_column}} %in% ADHD_U) ~ "Unspecified",
        TRUE ~ "None"
      )) %>%
   summarize("ADHD_subtype" = first(ADHD), .groups = "drop")
}

# Define function: extract_substance_dependence()
extract_substance_dependence <- function(data, id_column, dx_column) {

  data %>%
    select({{id_column}}, {{dx_column}}) %>%
    distinct() %>%
    group_by({{id_column}}) %>%
    mutate(
      alcohol_dependence = str_detect({{dx_column}},
                              regex("alcohol dependence|alcoholism|alcohol abuse",
                              ignore_case = TRUE)
                            ),
      cannabis_dependence = str_detect({{dx_column}},
                              regex("cannabis dependence",
                              ignore_case = TRUE)
                            ),
      cocaine_dependence = str_detect({{dx_column}},
                              regex("cocaine dependence",
                              ignore_case = TRUE)
                            ),
      nicotine_dependence = str_detect({{dx_column}},
                              regex("nicotine dependence|tobacco dependence",
                              ignore_case = TRUE)
                            ),
      opioid_dependence = str_detect({{dx_column}},
                              regex("heroin dependence|opioid dependence",
                              ignore_case = TRUE)
                            )
    ) %>%
    select({{id_column}}, alcohol_dependence:opioid_dependence) %>%
    summarize(
      across(alcohol_dependence:opioid_dependence,
         ~any(.x)), 
         .groups = "drop")
}

In [ ]:
# Extract ADHD dxs
adhd <- extract_ADHD_subtype(condition_df, person_id, standard_concept_name)

# Extract dependence dxs
dependence <- extract_substance_dependence(condition_df, person_id, standard_concept_name)

In [ ]:
head(adhd)
head(dependence)
nrow(adhd)
nrow(dependence)

In [ ]:
# Merge with demographics_df

merge1 <- left_join(adhd, dependence, by = 'person_id')
cohort_full <- left_join(demographics, merge1, by = 'person_id')

In [ ]:
# Remove unneeded columns
cohort_full <- cohort_full %>%
    select(-distance)

In [ ]:
# Save output to bucket
my_dataframe <- cohort_full

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- '011626_cohort_full.csv'

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


## Descriptive analysis (NOT STARTED)

Consider including zip data in descriptive analysis

## Process dfs for substance use PheWAS (WIP)

In [3]:
library(tidyverse)
# Load data
name_of_file_in_bucket <- '011626_cohort_full.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
cohort_full  <- read_csv(name_of_file_in_bucket)
head(cohort_full)

character(0)

Rows: 35915 Columns: 18
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): gender, race, ethnicity, sex_at_birth, self_reported_category, grou...
dbl (6): person_id, exposure, birth_year, sex_binary, weights, subclass
lgl (5): alcohol_dependence, cannabis_dependence, cocaine_dependence, nicoti...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


person_id,gender,race,ethnicity,sex_at_birth,self_reported_category,group,exposure,birth_year,sex_binary,weights,subclass,ADHD_subtype,alcohol_dependence,cannabis_dependence,cocaine_dependence,nicotine_dependence,opioid_dependence
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
7728356,Nonbinary,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,1991,0,1,1,Unspecified,FALSE,FALSE,FALSE,FALSE,FALSE
4411289,Transgender,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,2003,0,1,2,Combined,FALSE,TRUE,FALSE,TRUE,FALSE
2186562,Female,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,1996,0,1,3,Unspecified,FALSE,FALSE,FALSE,FALSE,FALSE
3041123,Female,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,1974,0,1,4,Combined,FALSE,FALSE,FALSE,TRUE,FALSE
3503287,Female,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,1957,0,1,5,Unspecified,FALSE,FALSE,FALSE,TRUE,FALSE
6769809,Female,American Indian/Alaska Native,Hispanic/Latino,Female,Multiple,ADHD,1,1961,0,1,6,Inattentive,TRUE,FALSE,FALSE,FALSE,FALSE


In [ ]:
# Alcohol dependence cohort for PheWAS
cohort_alcohol <- cohort_full %>%
    filter(alcohol_dependence == TRUE)
nrow(cohort_alcohol)

In [ ]:
# Cannabis dependence cohort for PheWAS
cohort_cannabis <- cohort_full %>%
    filter(cannabis_dependence == TRUE)
nrow(cohort_cannabis)

In [ ]:
# Cocaine dependence cohort for PheWAS
cohort_cocaine <- cohort_full %>%
    filter(cocaine_dependence == TRUE)
nrow(cohort_cocaine)

In [ ]:
# Nicotine dependence cohort for PheWAS
cohort_nicotine <- cohort_full %>%
    filter(nicotine_dependence == TRUE)
nrow(cohort_nicotine)

In [9]:
# Opioid dependence cohort for PheWAS
cohort_opioid <- cohort_full %>%
    filter(opioid_dependence == TRUE)
nrow(cohort_opioid)

[1] 1442

In [21]:
# No dependence cohort for PheWAS
cohort_no_sud <- cohort_full %>%
    mutate(n_dependence = rowSums(across(alcohol_dependence:opioid_dependence), na.rm = TRUE)) %>%
    filter(n_dependence == 0)

In [22]:
sum(is.na(cohort_no_sud$n_dependence))

[1] 0

In [24]:
# Save dependence cohorts to bucket
# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- cohort_no_sud

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- '011626_cohort_no_sud.csv'

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


character(0)

[1] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_condition_df.csv"   
 [2] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_person_df.csv"      
 [3] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_zip_df.csv"         
 [4] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_alcohol.csv"      
 [5] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_cannabis.csv"     
 [6] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_cocaine.csv"      
 [7] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_full.csv"         
 [8] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_nicotine.csv"     
 [9] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_no_sud.csv"       
[10] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_cohort_opioid.csv"       
[11] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_condition_df.csv"
[12] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_person_df.csv"   
[13] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_zip_df.csv"      
[14] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_demographics.csv"        
[15] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_SUD_phewas_df.csv"         
[16] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_SUD_tidy_df.csv"           
[17] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_condition_df.csv"          
[18] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_conditions.csv"            
[19] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_demographics.csv"          
[20] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_person_df.csv"             
[21] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_survey_df.csv"             
[22] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_varSUD_phewas_df.csv"      
[23] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/ADHD_zip_df.csv"                
[24] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/NO_ADHD_conditions_matched1.csv"
[25] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/NO_ADHD_demographics_full.csv"  
[26] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/NO_ADHD_person_df.csv"          
[27] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/SUD_phewas_df.csv"              
[28] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/anon_condition_df.csv"          
[29] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/phewas_counts.csv"              
[30] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/phewas_results_011626.csv"      
[31] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/phewas_results_ADHD_varSUD.csv" 
[32] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/phewas_results_adhd.csv"        
[33] "gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/phewas_results_sud.csv"

## Archive

In [ ]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
system(paste0("gsutil ls -r ", my_bucket), intern=T)